In [6]:
import os
import csv
import time
import requests
from PIL import Image
from io import BytesIO

OUTPUT_DIR = "negative_128_inaturalist"
SIZE = 128
PER_PAGE = 100
MAX_IMAGES = 3000

API_URL = "https://api.inaturalist.org/v1/observations"

HEADERS = {
    "User-Agent": "parakeet-negative-dataset/1.0"
}

# Tout ça doit rester NEGATIF : pas de perruche verte
TAXA = [
    "bird feeder",
    "sparrow",
    "pigeon",
    "crow",
    "robin",
    "blackbird",
    "finch",
    "starling",
    "tit",
    "dove",
    "squirrel",
    "tree",
    "branch",
    "leaf",
    "green plant",
    "bush",
    "shrub",
]

PHOTO_LICENSES = "cc0,cc-by,cc-by-nc"

os.makedirs(OUTPUT_DIR, exist_ok=True)


def safe_get(url, params=None, max_retries=5):
    for attempt in range(max_retries):
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=30)

            if r.status_code == 429:
                wait = 10 * (attempt + 1)
                print(f"429 Too Many Requests — pause {wait}s")
                time.sleep(wait)
                continue

            if r.status_code in [403, 404]:
                return None

            r.raise_for_status()
            return r

        except Exception as e:
            wait = 5 * (attempt + 1)
            print(f"Erreur réseau : {e} — pause {wait}s")
            time.sleep(wait)

    return None


def resize_and_save(image_bytes, filename):
    img = Image.open(BytesIO(image_bytes)).convert("RGB")

    w, h = img.size
    side = min(w, h)
    left = (w - side) // 2
    top = (h - side) // 2

    img = img.crop((left, top, left + side, top + side))
    img = img.resize((SIZE, SIZE), Image.Resampling.LANCZOS)

    path = os.path.join(OUTPUT_DIR, filename)
    img.save(path, "JPEG", quality=85)


def get_photo_url(photo):
    url = photo.get("url")
    if not url:
        return None

    # iNaturalist fournit souvent square.jpg ; on garde une petite taille.
    return url.replace("square.", "medium.")


count = 0
seen_urls = set()

metadata_path = os.path.join(OUTPUT_DIR, "metadata.csv")

with open(metadata_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "filename",
        "taxon_query",
        "observed_taxon",
        "photo_url",
        "license_code",
        "attribution",
        "observation_url"
    ])

    for taxon in TAXA:
        if count >= MAX_IMAGES:
            break

        print(f"\nRecherche : {taxon}")

        page = 1

        while count < MAX_IMAGES:
            params = {
                "q": taxon,
                "photos": "true",
                "photo_license": PHOTO_LICENSES,
                "per_page": PER_PAGE,
                "page": page,
                "order_by": "created_at",
                "order": "desc",
            }

            r = safe_get(API_URL, params=params)
            if r is None:
                break

            data = r.json()
            results = data.get("results", [])

            if not results:
                break

            for obs in results:
                if count >= MAX_IMAGES:
                    break

                photos = obs.get("photos", [])
                if not photos:
                    continue

                photo = photos[0]
                photo_url = get_photo_url(photo)

                if not photo_url or photo_url in seen_urls:
                    continue

                seen_urls.add(photo_url)

                img_response = safe_get(photo_url)
                if img_response is None:
                    continue

                filename = f"negative_{count + 1:05d}.jpg"

                try:
                    resize_and_save(img_response.content, filename)

                    taxon_name = obs.get("taxon", {}).get("name", "")
                    obs_url = obs.get("uri", "")
                    license_code = photo.get("license_code", "")
                    attribution = photo.get("attribution", "")

                    writer.writerow([
                        filename,
                        taxon,
                        taxon_name,
                        photo_url,
                        license_code,
                        attribution,
                        obs_url
                    ])

                    count += 1
                    print(f"{count} images téléchargées")

                except Exception as e:
                    print(f"Erreur image : {e}")

                time.sleep(0.2)

            page += 1
            time.sleep(1)

print(f"\nTerminé : {count} images dans {OUTPUT_DIR}/")
print(f"Métadonnées : {metadata_path}")


Recherche : bird feeder
1 images téléchargées
2 images téléchargées
3 images téléchargées
Erreur image : 'NoneType' object has no attribute 'get'
4 images téléchargées
5 images téléchargées
6 images téléchargées
7 images téléchargées
8 images téléchargées
9 images téléchargées
10 images téléchargées
11 images téléchargées
12 images téléchargées
13 images téléchargées
14 images téléchargées
15 images téléchargées
16 images téléchargées
17 images téléchargées
18 images téléchargées
19 images téléchargées
20 images téléchargées
21 images téléchargées
22 images téléchargées
23 images téléchargées
24 images téléchargées
25 images téléchargées
26 images téléchargées
27 images téléchargées
28 images téléchargées
29 images téléchargées
30 images téléchargées
31 images téléchargées
32 images téléchargées
33 images téléchargées
34 images téléchargées
35 images téléchargées
36 images téléchargées
37 images téléchargées
38 images téléchargées
39 images téléchargées
40 images téléchargées
41 image

In [7]:
import os
import csv
import time
import requests
from PIL import Image
from io import BytesIO

OUTPUT_DIR = "positive_green_parakeets_128"
SIZE = 128
PER_PAGE = 100
MAX_IMAGES = 300

API_URL = "https://api.inaturalist.org/v1/observations"

HEADERS = {
    "User-Agent": "green-parakeet-dataset-builder/1.0"
}

# Requêtes positives
TAXA = [
    "budgerigar",
    "green budgerigar",
    "parakeet",
    "green parakeet",
    "ring-necked parakeet",
    "rose-ringed parakeet",
    "monk parakeet",
    "green parrot",
]

PHOTO_LICENSES = "cc0,cc-by,cc-by-nc"

os.makedirs(OUTPUT_DIR, exist_ok=True)


def safe_get(url, params=None, max_retries=5):
    for attempt in range(max_retries):
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=30)

            if r.status_code == 429:
                wait = 10 * (attempt + 1)
                print(f"429 Too Many Requests — pause {wait}s")
                time.sleep(wait)
                continue

            if r.status_code in [403, 404]:
                return None

            r.raise_for_status()
            return r

        except Exception as e:
            wait = 5 * (attempt + 1)
            print(f"Erreur réseau : {e} — pause {wait}s")
            time.sleep(wait)

    return None


def get_photo_url(photo):
    url = photo.get("url")
    if not url:
        return None

    # medium donne une meilleure base avant resize en 128x128
    return url.replace("square.", "medium.")


def resize_and_save(image_bytes, filename):
    img = Image.open(BytesIO(image_bytes)).convert("RGB")

    w, h = img.size
    side = min(w, h)

    left = (w - side) // 2
    top = (h - side) // 2

    img = img.crop((left, top, left + side, top + side))
    img = img.resize((SIZE, SIZE), Image.Resampling.LANCZOS)

    path = os.path.join(OUTPUT_DIR, filename)
    img.save(path, "JPEG", quality=85)


count = 0
seen_urls = set()

metadata_path = os.path.join(OUTPUT_DIR, "metadata.csv")

with open(metadata_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "filename",
        "query",
        "observed_taxon",
        "photo_url",
        "license_code",
        "attribution",
        "observation_url"
    ])

    for taxon in TAXA:
        if count >= MAX_IMAGES:
            break

        print(f"\nRecherche : {taxon}")
        page = 1

        while count < MAX_IMAGES:
            params = {
                "q": taxon,
                "photos": "true",
                "photo_license": PHOTO_LICENSES,
                "per_page": PER_PAGE,
                "page": page,
                "order_by": "created_at",
                "order": "desc",
            }

            r = safe_get(API_URL, params=params)
            if r is None:
                break

            data = r.json()
            results = data.get("results", [])

            if not results:
                break

            for obs in results:
                if count >= MAX_IMAGES:
                    break

                photos = obs.get("photos", [])
                if not photos:
                    continue

                photo = photos[0]
                photo_url = get_photo_url(photo)

                if not photo_url or photo_url in seen_urls:
                    continue

                seen_urls.add(photo_url)

                img_response = safe_get(photo_url)
                if img_response is None:
                    continue

                filename = f"positive_{count + 1:05d}.jpg"

                try:
                    resize_and_save(img_response.content, filename)

                    taxon_name = obs.get("taxon", {}).get("name", "")
                    obs_url = obs.get("uri", "")
                    license_code = photo.get("license_code", "")
                    attribution = photo.get("attribution", "")

                    writer.writerow([
                        filename,
                        taxon,
                        taxon_name,
                        photo_url,
                        license_code,
                        attribution,
                        obs_url
                    ])

                    count += 1
                    print(f"{count} images positives téléchargées")

                except Exception as e:
                    print(f"Erreur image : {e}")

                time.sleep(0.2)

            page += 1
            time.sleep(1)

print(f"\nTerminé : {count} images dans {OUTPUT_DIR}/")
print(f"Métadonnées : {metadata_path}")


Recherche : budgerigar
1 images positives téléchargées
2 images positives téléchargées
3 images positives téléchargées
4 images positives téléchargées
5 images positives téléchargées
6 images positives téléchargées
7 images positives téléchargées
8 images positives téléchargées
9 images positives téléchargées
10 images positives téléchargées
11 images positives téléchargées
12 images positives téléchargées
13 images positives téléchargées
14 images positives téléchargées
15 images positives téléchargées
16 images positives téléchargées
17 images positives téléchargées
18 images positives téléchargées
19 images positives téléchargées
20 images positives téléchargées
21 images positives téléchargées
22 images positives téléchargées
23 images positives téléchargées
24 images positives téléchargées
25 images positives téléchargées
26 images positives téléchargées
27 images positives téléchargées
28 images positives téléchargées
29 images positives téléchargées
30 images positives téléchar

In [10]:
from pathlib import Path
from PIL import Image
import string

# Dossiers
INPUT_DIR = Path("./photos_perruches")
OUTPUT_DIR = Path("./photos_perruches_128")

OUTPUT_DIR.mkdir(exist_ok=True)

SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp", ".tiff"}


def number_to_letters(n):
    """0 -> a, 1 -> b, ..., 25 -> z, 26 -> aa, 27 -> ab, ..."""
    letters = string.ascii_lowercase
    result = ""

    while True:
        result = letters[n % 26] + result
        n = n // 26 - 1
        if n < 0:
            break

    return result


# Tri des fichiers pour avoir un ordre reproductible
image_files = sorted(
    [f for f in INPUT_DIR.iterdir() if f.suffix.lower() in SUPPORTED_EXTENSIONS]
)

for idx, image_path in enumerate(image_files):
    try:
        with Image.open(image_path) as img:
            img = img.convert("RGB")

            # Redimensionnement à 128x128
            resized = img.resize((128, 128), Image.Resampling.LANCZOS)

            # Nouveau nom
            new_name = f"{number_to_letters(idx)}.png"
            output_path = OUTPUT_DIR / new_name

            resized.save(output_path)

            print(f"✓ {image_path.name} -> {new_name}")

    except Exception as e:
        print(f"✗ Erreur sur {image_path.name}: {e}")

print(f"\nTerminé : {len(image_files)} images traitées.")

✓ 1.jpg -> a.png
✓ 10.jpg -> b.png
✓ 11.jpg -> c.png
✓ 12.jpg -> d.png
✓ 13.jpg -> e.png
✓ 14.jpeg -> f.png
✓ 15.jpeg -> g.png
✓ 16.jpg -> h.png
✓ 17.jpg -> i.png
✓ 18.jpg -> j.png
✓ 19.jpg -> k.png
✓ 2.jpeg -> l.png
✓ 20.jpeg -> m.png
✓ 21.jpeg -> n.png
✓ 22.jpg -> o.png
✓ 23.jpg -> p.png
✓ 24.jpeg -> q.png
✓ 25.jpeg -> r.png
✓ 26.jpg -> s.png
✓ 27.jpg -> t.png
✓ 28.jpeg -> u.png
✓ 29.jpeg -> v.png
✓ 3.jpg -> w.png
✓ 30.jpeg -> x.png
✓ 31.jpeg -> y.png
✓ 32.jpg -> z.png
✓ 33.jpg -> aa.png
✓ 34.jpg -> ab.png
✓ 35.jpeg -> ac.png
✓ 36.jpeg -> ad.png
✓ 37.jpg -> ae.png
✓ 38.jpeg -> af.png
✓ 39.jpg -> ag.png
✓ 4.jpg -> ah.png
✓ 40.jpg -> ai.png
✓ 41.jpg -> aj.png
✓ 42.jpg -> ak.png
✓ 43.jpeg -> al.png
✓ 44.jpeg -> am.png
✓ 45.jpeg -> an.png
✓ 46.jpeg -> ao.png
✓ 47.jpg -> ap.png
✓ 48.jpeg -> aq.png
✓ 49.jpeg -> ar.png
✓ 5.jpg -> as.png
✓ 50.jpeg -> at.png
✓ 51.jpg -> au.png
✓ 52.jpg -> av.png
✓ 53.jpeg -> aw.png
✓ 54.jpg -> ax.png
✓ 55.jpeg -> ay.png
✓ 56.jpeg -> az.png
✓ 57.jpg -> ba.png


In [9]:
import shutil
from pathlib import Path

OUTPUT_DIR = Path("./photos_perruches_128")

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
    print(f"Content of {OUTPUT_DIR} deleted.")

OUTPUT_DIR.mkdir(exist_ok=True)
print(f"Directory {OUTPUT_DIR} recreated.")

Content of photos_perruches_128 deleted.
Directory photos_perruches_128 recreated.
